In [5]:
import pandas as pd
import pyodbc
import warnings

# Suppress standard library warnings for a cleaner output grid
warnings.filterwarnings('ignore')

# 1. Establish the Enterprise Bridge Coordinates
# The 'r' before the string ensures Python reads the backslash correctly
conn_str = (
    r"DRIVER={SQL Server};"
    r"SERVER=localhost\SQLEXPRESS;"
    r"DATABASE=FactoryOperations;"
    r"Trusted_Connection=yes;"
)

# 2. Open the server connection
print("Initiating server handshake...")
connection = pyodbc.connect(conn_str)

# 3. Define the SQL Query (Your exact, verified syntax)
sql_query = """
SELECT
    TableA.machine_id,
    TableB.manufacturer,
    TableB.procurement_year,
    TableA.defect_count
FROM production_logs AS TableA
INNER JOIN machine_assets AS TableB
    ON TableA.machine_id = TableB.asset_id;
"""

# 4. Execute the query and ingest directly into a Pandas DataFrame
print("Executing relational query and ingesting data...")
df_root_cause = pd.read_sql(sql_query, connection)

# 5. Sever the connection (Standard security and memory best practice)
connection.close()
print("Connection securely closed.\n")

# 6. Display the resulting Python Data Engine
print("--- DATAFRAME ACQUIRED ---")
print(df_root_cause)
# 1. Export the SQL-derived DataFrame directly to an Excel file
file_name = "Munich_Root_Cause_Analysis.xlsx"
df_root_cause.to_excel(file_name, index=False)

print(f"Pipeline Complete: Data successfully exported to {file_name}")

Initiating server handshake...
Executing relational query and ingesting data...
Connection securely closed.

--- DATAFRAME ACQUIRED ---
  machine_id manufacturer  procurement_year  defect_count
0     CNC-01      Siemens              2018             2
1     CNC-02      Siemens              2015            15
2   Press-01        Bosch              2024             0
3   Lathe-03         Haas              2020             5
4    Oven-01   Thermcraft              2012             1
Pipeline Complete: Data successfully exported to Munich_Root_Cause_Analysis.xlsx


In [6]:
import pandas as pd
from sqlalchemy import create_engine
import warnings

# Suppress non-critical warnings
warnings.filterwarnings('ignore')

# --- PHASE 1: THE EXTRACTION (SQL) ---
print("[1/4] Establishing SQLAlchemy Engine...")
# The connection string syntax for SQLAlchemy utilizing your existing pyodbc driver
# Note the specific URI formatting required for SQL Server
conn_uri = "mssql+pyodbc://@localhost\\SQLEXPRESS/FactoryOperations?driver=SQL+Server&Trusted_Connection=yes"
engine = create_engine(conn_uri)

extraction_query = """
SELECT 
    TableA.machine_id, 
    TableB.manufacturer, 
    TableA.spindle_temp, 
    TableA.defect_count
FROM production_logs AS TableA
INNER JOIN machine_assets AS TableB
    ON TableA.machine_id = TableB.asset_id;
"""

print("[2/4] Executing Relational Extraction...")
# Pandas utilizes the SQLAlchemy engine to ingest the data seamlessly
df_raw = pd.read_sql(extraction_query, engine)


# --- PHASE 2: THE TRANSFORMATION (PANDAS) ---
print("[3/4] Executing Analytical Transformations...")
# 1. Filter: Isolate machinery exceeding the 80.0 degree threshold
df_critical = df_raw[df_raw['spindle_temp'] > 80.0].copy()

# 2. Enhance: Append a dynamic alert column
df_critical['Thermal_Status'] = 'CRITICAL ALERT'


# --- PHASE 3: THE LOAD (EXCEL EXPORT) ---
print("[4/4] Generating Executive Excel Report...")
report_filename = "Critical_Thermal_Audit.xlsx"
df_critical.to_excel(report_filename, index=False)

print(f"\n--- PIPELINE COMPLETE ---")
print(f"Data successfully transformed and exported to: {report_filename}")
print(df_critical)

[1/4] Establishing SQLAlchemy Engine...
[2/4] Executing Relational Extraction...
[3/4] Executing Analytical Transformations...
[4/4] Generating Executive Excel Report...

--- PIPELINE COMPLETE ---
Data successfully transformed and exported to: Critical_Thermal_Audit.xlsx
  machine_id manufacturer  spindle_temp  defect_count  Thermal_Status
1     CNC-02      Siemens          82.1            15  CRITICAL ALERT
4    Oven-01   Thermcraft         110.5             1  CRITICAL ALERT
